# PRISM Causal Forest Modeling

This notebook is the report-ready causal forest modeling workflow for the PRISM intervention benefit project. It follows the same broad order as `PRISM_Intervention_Benefit_Modeling_README.md`, but adapts Analytical Task 3 for causal forest diagnostics rather than factual outcome-model performance.

Core sign convention:

```text
tau_hat = estimated effect of intervention on outcome_ed_90d
benefit_score = -tau_hat
higher benefit_score = larger estimated ED risk reduction from intervention
```

The workflow uses the same reproducibility seed as the T-learner and X-learner workflow: `123`.

## Optional Package Install

Run this cell only if the current notebook kernel is missing `econml`. A Python 3.10-3.12 environment is recommended because `econml` may not install cleanly on newer Python versions.

In [ ]:
# Uncomment if needed in a compatible Python environment.
# %pip install econml scikit-learn pandas numpy matplotlib openpyxl

## Background

Care management programs must decide which members should receive intervention when outreach resources are limited. A common approach is to prioritize the highest-risk members, but high baseline risk does not always mean high intervention benefit. This causal forest workflow focuses on estimating whether intervention benefit varies across members, also called heterogeneous treatment effect, or HTE.

## Business Question

Which members are most likely to benefit from intervention in terms of reducing 90-day emergency department utilization, based on causal forest estimates of heterogeneous treatment effects?

## Project Objectives

- Estimate member-level heterogeneous treatment effects.
- Rank members by estimated intervention benefit.
- Identify high-benefit HTE deciles and subgroup profiles.
- Compare causal forest rankings with existing T-learner and X-learner outputs where possible.
- Provide a partial explainability layer through causal forest variable importance.
- Produce README-ready CSV tables and charts.

## Analytical Task 1: Understanding And Explaining The Causal Forest Framework

The causal forest model estimates a treatment effect for each member. In this project, the outcome is `outcome_ed_90d`, and the treatment is `intervention_flag`. Since ED utilization is an undesirable outcome, a negative `tau_hat` means intervention is estimated to reduce ED risk. For business interpretation, this notebook reports `benefit_score = -tau_hat`, so higher values mean larger estimated benefit.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except Exception:
    display = print

if importlib.util.find_spec('econml') is None:
    raise ImportError('Missing required package: econml. Install it in a Python 3.10-3.12 environment, then rerun this notebook.')

from econml.dml import CausalForestDML

CODE_DIR = Path.cwd()
if CODE_DIR.name.lower() != 'code':
    CODE_DIR = Path.cwd() / 'Code'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from _prism_model_utils import (
    add_date_features,
    clean_names_simple,
    ensure_output_folder,
    make_design_matrix,
    ntile_desc,
    prepare_model_frame,
    read_prism_excel,
    require_columns,
    split_train_test,
    to_binary,
)

PROJECT_ROOT = CODE_DIR.parent
SEED = 123
TRAIN_FRACTION = 0.70
OUTCOME_COL = 'outcome_ed_90d'
TREATMENT_COL = 'intervention_flag'
OUTPUT_DIR = ensure_output_folder(PROJECT_ROOT / 'Outputs' / 'Causal-Forests' / 'Python')

np.random.seed(SEED)
warnings.filterwarnings('ignore', category=UserWarning)

print(f'Project root: {PROJECT_ROOT}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Seed: {SEED}')

In [ ]:
PATHS = {
    'scored_output': OUTPUT_DIR / 'causal_forest_scored_output.csv',
    'test_scored_output': OUTPUT_DIR / 'causal_forest_scored_test_output.csv',
    'data_review_summary': OUTPUT_DIR / 'causal_forest_data_review_summary.csv',
    'predictor_inventory': OUTPUT_DIR / 'causal_forest_predictor_inventory.csv',
    'event_count_summary': OUTPUT_DIR / 'causal_forest_event_count_summary.csv',
    'propensity_summary': OUTPUT_DIR / 'causal_forest_propensity_summary.csv',
    'effect_distribution_summary': OUTPUT_DIR / 'causal_forest_effect_distribution_summary.csv',
    'uncertainty_summary': OUTPUT_DIR / 'causal_forest_uncertainty_summary.csv',
    'ate_summary': OUTPUT_DIR / 'causal_forest_ate_summary.csv',
    'decile_summary': OUTPUT_DIR / 'causal_forest_decile_summary.csv',
    'top_benefit_examples': OUTPUT_DIR / 'causal_forest_top_benefit_examples.csv',
    'variable_importance': OUTPUT_DIR / 'causal_forest_variable_importance.csv',
    'top_decile_profile': OUTPUT_DIR / 'causal_forest_top_decile_profile.csv',
    'targeting_summary': OUTPUT_DIR / 'causal_forest_targeting_summary.csv',
    'consistency_summary': OUTPUT_DIR / 'causal_forest_vs_uplift_consistency_summary.csv',
    'propensity_chart': OUTPUT_DIR / 'dashboard_propensity_overlap.png',
    'effect_distribution_chart': OUTPUT_DIR / 'dashboard_causal_forest_effect_distribution.png',
    'benefit_decile_chart': OUTPUT_DIR / 'dashboard_causal_forest_avg_benefit_by_decile.png',
    'tau_decile_chart': OUTPUT_DIR / 'dashboard_causal_forest_tau_by_decile.png',
    'risk_benefit_decile_chart': OUTPUT_DIR / 'dashboard_causal_forest_risk_vs_benefit_by_decile.png',
    'variable_importance_chart': OUTPUT_DIR / 'dashboard_causal_forest_variable_importance.png',
    'comparison_chart': OUTPUT_DIR / 'dashboard_causal_forest_vs_uplift_comparison.png',
}

def save_csv(df, path):
    df.to_csv(path, index=False)
    print(f'Saved: {path}')

def save_current_figure(path):
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()
    print(f'Saved: {path}')

def effect_standard_errors(cf_model, x_matrix):
    try:
        inference = cf_model.effect_inference(x_matrix)
        return np.asarray(inference.stderr, dtype=float)
    except Exception as exc:
        print(f'Standard errors not available: {exc}')
        return np.full(len(x_matrix), np.nan)

def summarize_distribution(values, label):
    series = pd.Series(values, dtype=float).dropna()
    return pd.DataFrame({
        'metric': ['mean', 'std_dev', 'min', 'p10', 'p25', 'median', 'p75', 'p90', 'max'],
        label: [series.mean(), series.std(), series.min(), series.quantile(0.10), series.quantile(0.25), series.median(), series.quantile(0.75), series.quantile(0.90), series.max()],
    })

def safe_corr(a, b, method):
    joined = pd.concat([pd.Series(a, dtype=float), pd.Series(b, dtype=float)], axis=1).dropna()
    if len(joined) < 3:
        return np.nan
    return joined.iloc[:, 0].corr(joined.iloc[:, 1], method=method)

def top_overlap(a_scores, b_scores, share=0.10):
    a = pd.Series(a_scores).reset_index(drop=True)
    b = pd.Series(b_scores).reset_index(drop=True)
    n = min(len(a), len(b))
    if n == 0:
        return np.nan
    k = max(1, int(np.floor(n * share)))
    return len(set(a.iloc[:n].nlargest(k).index) & set(b.iloc[:n].nlargest(k).index)) / k

def propensity_auc(y_true, scores):
    try:
        return roc_auc_score(y_true, scores)
    except Exception:
        return np.nan

def fit_shared_propensity_model(x_matrix, treatment, seed=123):
    """Match the uplift/X-learner propensity model specification."""
    treatment_array = np.asarray(treatment, dtype=float)
    class_counts = pd.Series(treatment_array).value_counts()
    folds = int(min(5, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two treatment classes with at least two rows each for propensity modeling.')
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    propensity_model = LogisticRegressionCV(
        Cs=np.logspace(-4, 4, 30),
        cv=cv,
        penalty='elasticnet',
        solver='saga',
        l1_ratios=[0.5],
        scoring='roc_auc',
        max_iter=10000,
        random_state=seed,
        refit=True,
    )
    pipeline = make_pipeline(StandardScaler(), propensity_model)
    pipeline.fit(x_matrix, treatment_array)
    return pipeline

def clipped_propensity(propensity_model, x_matrix, lower=0.05, upper=0.95):
    propensity = propensity_model.predict_proba(x_matrix)[:, 1]
    return np.clip(propensity, lower, upper)

def load_saved_xlearner_test_propensity(expected_rows):
    """Reuse exact saved X-learner test propensities when row alignment is available."""
    xlearner_path = PROJECT_ROOT / 'Outputs' / 'Uplift' / 'Python' / 'X-Learner' / 'GLMNet' / 'xlearner_scored_test_output.csv'
    if not xlearner_path.exists():
        return None
    xlearner_df = pd.read_csv(xlearner_path)
    if len(xlearner_df) != expected_rows or 'xlearner_propensity_score' not in xlearner_df.columns:
        return None
    return xlearner_df['xlearner_propensity_score'].to_numpy(dtype=float)

## Analytical Task 2: Data Review

In [ ]:
df = read_prism_excel()
df.columns = clean_names_simple(df.columns)
df = df.copy()

require_columns(df, [OUTCOME_COL, TREATMENT_COL])
df[OUTCOME_COL] = to_binary(df[OUTCOME_COL])
df[TREATMENT_COL] = to_binary(df[TREATMENT_COL])
df = add_date_features(df, include_duration=False)

PREDICTOR_CATEGORIES = {
    'demographics': ['client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender', 'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag'],
    'clinical_conditions': ['diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag', 'behavioral_health_risk_flag'],
    'sdoh': ['food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag', 'utilities_insecurity_flag'],
    'utilization': ['pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m'],
    'pharmacy': ['total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag', 'opioid_flag', 'polypharmacy_flag'],
    'risk_scores': ['percolator_utilization_score', 'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'risk_tier'],
}
PREDICTOR_VARS = [feature for features in PREDICTOR_CATEGORIES.values() for feature in features]
NUMERIC_VARS = ['age', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'percolator_utilization_score', 'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'intervention_start_month', 'intervention_start_wday', 'days_to_intervention_start']
BINARY_EXTRA = ['dual_eligible', 'living_alone_flag']
present_predictors = [feature for feature in PREDICTOR_VARS if feature in df.columns]

predictor_inventory = pd.DataFrame([
    {'feature': feature, 'category': category, 'included_in_model': feature in present_predictors, 'reason_if_excluded': '' if feature in present_predictors else 'Column not present in source data', 'source_dtype': str(df[feature].dtype) if feature in df.columns else 'missing', 'unique_values': df[feature].nunique(dropna=True) if feature in df.columns else 0}
    for category, features in PREDICTOR_CATEGORIES.items()
    for feature in features
])
save_csv(predictor_inventory, PATHS['predictor_inventory'])

model_df = prepare_model_frame(df, present_predictors, NUMERIC_VARS, BINARY_EXTRA)
model_df.insert(0, 'row_id', np.arange(len(model_df)))
feature_cols_raw = [col for col in model_df.columns if col not in ['row_id', OUTCOME_COL, TREATMENT_COL]]

continuous_count = len([col for col in feature_cols_raw if col in NUMERIC_VARS])
binary_count = len([col for col in feature_cols_raw if col.endswith('_flag') or col in BINARY_EXTRA])
categorical_count = len(feature_cols_raw) - continuous_count - binary_count

data_review_summary = pd.DataFrame({
    'metric': ['Total members', 'Treated members', 'Untreated/control members', 'Treatment rate', 'ED outcome events', 'Outcome prevalence', 'Treated observed ED rate', 'Control observed ED rate', 'Final predictors before one-hot encoding', 'Continuous/count numeric predictors', 'Binary indicator predictors', 'Multi-level categorical predictors'],
    'current_value': [len(model_df), int((model_df[TREATMENT_COL] == 1).sum()), int((model_df[TREATMENT_COL] == 0).sum()), model_df[TREATMENT_COL].mean(), int((model_df[OUTCOME_COL] == 1).sum()), model_df[OUTCOME_COL].mean(), model_df.loc[model_df[TREATMENT_COL] == 1, OUTCOME_COL].mean(), model_df.loc[model_df[TREATMENT_COL] == 0, OUTCOME_COL].mean(), len(feature_cols_raw), continuous_count, binary_count, categorical_count]
})
save_csv(data_review_summary, PATHS['data_review_summary'])
display(data_review_summary)

## Analytical Task 3: Causal Forest Diagnostics And Estimation Credibility

This section replaces factual outcome-model performance. It checks event counts, treatment overlap, treatment-effect distribution, and treatment-effect uncertainty.

To strengthen comparability with the X-learner, this notebook uses the same GLMNet-style propensity specification from the uplift notebook: `StandardScaler()` plus `LogisticRegressionCV()` with elastic-net penalty, `l1_ratios=[0.5]`, ROC-AUC cross-validation, clipping to `[0.05, 0.95]`, and `SEED = 123`. If the X-learner scored test output exists and is row-aligned, this notebook reuses the exact saved `xlearner_propensity_score` values for the test-set diagnostics and output.

In [ ]:
feature_frame = model_df.drop(columns=['row_id', OUTCOME_COL, TREATMENT_COL])
_, [x_all] = make_design_matrix([feature_frame])

train_df, test_df = split_train_test(model_df, train_fraction=TRAIN_FRACTION, seed=SEED, stratify_columns=[TREATMENT_COL, OUTCOME_COL])
x_train = x_all.loc[train_df.index].reset_index(drop=True)
x_test = x_all.loc[test_df.index].reset_index(drop=True)
y_train = train_df[OUTCOME_COL].astype(float).to_numpy()
w_train = train_df[TREATMENT_COL].astype(float).to_numpy()
y_test = test_df[OUTCOME_COL].astype(float).to_numpy()
w_test = test_df[TREATMENT_COL].astype(float).to_numpy()

data_review_summary = pd.concat([data_review_summary, pd.DataFrame({'metric': ['Model matrix columns after one-hot encoding', 'Train rows', 'Test rows'], 'current_value': [x_all.shape[1], len(train_df), len(test_df)]})], ignore_index=True)
save_csv(data_review_summary, PATHS['data_review_summary'])

event_rows = []
for split_name, frame in [('Train', train_df), ('Test', test_df)]:
    for group_value, group_label in [(1.0, 'Treated'), (0.0, 'Control')]:
        subset = frame[frame[TREATMENT_COL] == group_value]
        positive = int((subset[OUTCOME_COL] == 1).sum())
        n = int(len(subset))
        event_rows.append({'split': split_name, 'group': group_label, 'n': n, 'positive_ed_events': positive, 'negative_ed_events': n - positive, 'event_rate': positive / n if n else np.nan})
event_count_summary = pd.DataFrame(event_rows)
save_csv(event_count_summary, PATHS['event_count_summary'])
display(event_count_summary)

shared_propensity_model = fit_shared_propensity_model(x_train, w_train, seed=SEED)
train_propensity = clipped_propensity(shared_propensity_model, x_train)
test_propensity_from_model = clipped_propensity(shared_propensity_model, x_test)
all_propensity = clipped_propensity(shared_propensity_model, x_all)

# If the uplift/X-learner test output already exists and has the same row count,
# reuse those exact member-level propensity values for test diagnostics/output.
saved_xlearner_test_propensity = load_saved_xlearner_test_propensity(len(test_df))
if saved_xlearner_test_propensity is not None:
    test_propensity = saved_xlearner_test_propensity
    propensity_source = 'saved_xlearner_scored_test_output'
else:
    test_propensity = test_propensity_from_model
    propensity_source = 'causal_forest_matched_glmnet_model'
propensity_series = pd.Series(test_propensity, dtype=float)
propensity_summary = pd.DataFrame({'metric': ['Propensity source', 'Train treatment model AUC', 'Test treatment model AUC', 'Mean propensity', 'Min propensity', '5th percentile', 'Median propensity', '95th percentile', 'Max propensity', 'Members below 0.05', 'Members above 0.95'], 'value': [propensity_source, propensity_auc(w_train, train_propensity), propensity_auc(w_test, test_propensity), propensity_series.mean(), propensity_series.min(), propensity_series.quantile(0.05), propensity_series.median(), propensity_series.quantile(0.95), propensity_series.max(), int((propensity_series < 0.05).sum()), int((propensity_series > 0.95).sum())]})
save_csv(propensity_summary, PATHS['propensity_summary'])
display(propensity_summary)

plt.figure(figsize=(8, 4.5))
plt.hist(test_propensity[w_test == 1], bins=15, alpha=0.65, label='Treated')
plt.hist(test_propensity[w_test == 0], bins=15, alpha=0.65, label='Control')
plt.xlabel('Estimated propensity for intervention')
plt.ylabel('Members')
plt.title('Causal Forest Propensity Overlap Check')
plt.legend()
save_current_figure(PATHS['propensity_chart'])

## Fit Causal Forest Model

In [ ]:
causal_forest_treatment_model = make_pipeline(
    StandardScaler(),
    LogisticRegressionCV(
        Cs=np.logspace(-4, 4, 30),
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        penalty='elasticnet',
        solver='saga',
        l1_ratios=[0.5],
        scoring='roc_auc',
        max_iter=10000,
        random_state=SEED,
        refit=True,
    )
)

cf_model = CausalForestDML(
    model_y=RandomForestRegressor(n_estimators=300, min_samples_leaf=10, random_state=SEED, n_jobs=-1),
    model_t=causal_forest_treatment_model,
    discrete_treatment=True,
    n_estimators=800,
    min_samples_leaf=10,
    max_depth=None,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    random_state=SEED,
)
cf_model.fit(y_train, w_train, X=x_train)
print('Causal forest trained successfully.')

## Analytical Task 4: Treatment Effect Analysis

In [ ]:
tau_test = np.asarray(cf_model.effect(x_test), dtype=float)
tau_se_test = effect_standard_errors(cf_model, x_test)

results_test = test_df.reset_index(drop=True).copy()
results_test['tau_hat'] = tau_test
results_test['tau_se'] = tau_se_test
results_test['benefit_score'] = -results_test['tau_hat']
results_test['hte_decile'] = ntile_desc(results_test['benefit_score'], 10).to_numpy()
results_test['uplift_decile'] = results_test['hte_decile']
results_test['propensity_score'] = test_propensity
results_test['tau_ci_lower'] = results_test['tau_hat'] - 1.96 * results_test['tau_se']
results_test['tau_ci_upper'] = results_test['tau_hat'] + 1.96 * results_test['tau_se']
results_test['benefit_ci_lower'] = -results_test['tau_ci_upper']
results_test['benefit_ci_upper'] = -results_test['tau_ci_lower']
save_csv(results_test, PATHS['test_scored_output'])

ate_summary = pd.DataFrame({'metric': ['avg_tau_hat', 'avg_benefit_score', 'test_members'], 'value': [results_test['tau_hat'].mean(), results_test['benefit_score'].mean(), len(results_test)]})
save_csv(ate_summary, PATHS['ate_summary'])
display(ate_summary)

effect_distribution_summary = summarize_distribution(results_test['tau_hat'], 'tau_hat').merge(summarize_distribution(results_test['benefit_score'], 'benefit_score'), on='metric', how='outer')
save_csv(effect_distribution_summary, PATHS['effect_distribution_summary'])
display(effect_distribution_summary)

uncertainty_summary = pd.DataFrame({'metric': ['Mean tau standard error', 'Median tau standard error', 'Members with tau CI entirely below zero', 'Members with tau CI crossing zero', 'Members with tau CI entirely above zero', 'Top HTE decile mean tau standard error'], 'value': [results_test['tau_se'].mean(), results_test['tau_se'].median(), int((results_test['tau_ci_upper'] < 0).sum()), int(((results_test['tau_ci_lower'] <= 0) & (results_test['tau_ci_upper'] >= 0)).sum()), int((results_test['tau_ci_lower'] > 0).sum()), results_test.loc[results_test['hte_decile'] == 1, 'tau_se'].mean()]})
save_csv(uncertainty_summary, PATHS['uncertainty_summary'])
display(uncertainty_summary)

plt.figure(figsize=(8, 4.5))
plt.hist(results_test['benefit_score'], bins=20, alpha=0.85)
plt.axvline(results_test['benefit_score'].mean(), linestyle='--')
plt.xlabel('Benefit score (-tau_hat)')
plt.ylabel('Members')
plt.title('Causal Forest Estimated Benefit Distribution')
save_current_figure(PATHS['effect_distribution_chart'])

## Analytical Task 5: HTE Decile And High-Value Subgroup Analysis

In [ ]:
agg_map = {
    'n': ('hte_decile', 'size'),
    'avg_tau_hat': ('tau_hat', 'mean'),
    'avg_benefit_score': ('benefit_score', 'mean'),
    'avg_tau_se': ('tau_se', 'mean'),
    'observed_ed_rate': (OUTCOME_COL, 'mean'),
    'treatment_pct': (TREATMENT_COL, 'mean'),
    'avg_propensity_score': ('propensity_score', 'mean'),
}
if 'current_risk_score' in results_test.columns:
    agg_map['avg_current_risk_score'] = ('current_risk_score', 'mean')

decile_summary = results_test.groupby('hte_decile', as_index=False).agg(**agg_map).sort_values('hte_decile')
decile_summary['uplift_decile'] = decile_summary['hte_decile']
save_csv(decile_summary, PATHS['decile_summary'])
display(decile_summary)

plt.figure(figsize=(8, 4.5))
plt.bar(decile_summary['hte_decile'].astype(str), decile_summary['avg_benefit_score'])
plt.xlabel('HTE decile (1 = highest estimated benefit)')
plt.ylabel('Average benefit score')
plt.title('Causal Forest Average Estimated Benefit By HTE Decile')
save_current_figure(PATHS['benefit_decile_chart'])

plt.figure(figsize=(8, 4.5))
plt.bar(decile_summary['hte_decile'].astype(str), decile_summary['avg_tau_hat'])
plt.axhline(0, linewidth=1)
plt.xlabel('HTE decile (1 = highest estimated benefit)')
plt.ylabel('Average tau_hat')
plt.title('Causal Forest Average Treatment Effect By HTE Decile')
save_current_figure(PATHS['tau_decile_chart'])

In [ ]:
example_rows = []
ranked = results_test.sort_values('benefit_score', ascending=False)
example_rows.append(('Highest benefit', ranked.iloc[0], 'Strong outreach candidate based on estimated ED risk reduction.'))
example_rows.append(('Lowest benefit', ranked.iloc[-1], 'Lowest priority by causal forest benefit score.'))
if 'current_risk_score' in ranked.columns:
    high_risk = ranked['current_risk_score'].quantile(0.75)
    low_benefit = ranked['benefit_score'].quantile(0.25)
    subset = ranked[(ranked['current_risk_score'] >= high_risk) & (ranked['benefit_score'] <= low_benefit)]
    if not subset.empty:
        example_rows.append(('High risk, low benefit', subset.iloc[0], 'High baseline risk but limited estimated impactability.'))
    low_risk = ranked['current_risk_score'].quantile(0.50)
    high_benefit = ranked['benefit_score'].quantile(0.75)
    subset = ranked[(ranked['current_risk_score'] <= low_risk) & (ranked['benefit_score'] >= high_benefit)]
    if not subset.empty:
        example_rows.append(('Low risk, high benefit', subset.iloc[0], 'May be missed by risk-only targeting but appears impactable.'))

top_benefit_examples = pd.DataFrame([
    {'member_profile': label, 'row_id': int(row['row_id']), 'actual_outcome': row[OUTCOME_COL], 'treatment_flag': row[TREATMENT_COL], 'current_risk_score': row.get('current_risk_score', np.nan), 'tau_hat': row['tau_hat'], 'tau_se': row['tau_se'], 'benefit_score': row['benefit_score'], 'hte_decile': row['hte_decile'], 'outreach_interpretation': interpretation}
    for label, row, interpretation in example_rows
]).drop_duplicates(['member_profile', 'row_id'])
save_csv(top_benefit_examples, PATHS['top_benefit_examples'])
display(top_benefit_examples)

### Framework Consistency Check

In [ ]:
# Full-file scoring is needed before full T-learner comparison.
tau_full = np.asarray(cf_model.effect(x_all), dtype=float)
tau_se_full = effect_standard_errors(cf_model, x_all)
scored_full = model_df.copy()
scored_full['tau_hat'] = tau_full
scored_full['tau_se'] = tau_se_full
scored_full['benefit_score'] = -scored_full['tau_hat']
scored_full['hte_decile'] = ntile_desc(scored_full['benefit_score'], 10).to_numpy()
scored_full['uplift_decile'] = scored_full['hte_decile']
scored_full['propensity_score'] = all_propensity
save_csv(scored_full, PATHS['scored_output'])

comparison_rows = []
t_path = PROJECT_ROOT / 'Outputs' / 'Uplift' / 'Python' / 'T-Learner' / 'GLMNet' / 'uplift_scored_output.csv'
if t_path.exists():
    t_df = pd.read_csv(t_path)
    n = min(len(scored_full), len(t_df))
    t_scores = t_df['benefit_score'].iloc[:n] if 'benefit_score' in t_df.columns else pd.Series(np.nan, index=range(n))
    cf_scores = scored_full['benefit_score'].iloc[:n]
    comparison_rows.append({'comparison': 'Causal forest vs GLMNet T-learner full output', 'comparison_basis': 'row_order_full_file_no_stable_member_id', 'n_compared': n, 'pearson_corr': safe_corr(cf_scores, t_scores, 'pearson'), 'spearman_corr': safe_corr(cf_scores, t_scores, 'spearman'), 'top_decile_overlap': top_overlap(cf_scores, t_scores, 0.10), 'top_20pct_overlap': top_overlap(cf_scores, t_scores, 0.20)})

x_path = PROJECT_ROOT / 'Outputs' / 'Uplift' / 'Python' / 'X-Learner' / 'GLMNet' / 'xlearner_scored_test_output.csv'
if x_path.exists():
    x_df = pd.read_csv(x_path)
    n = min(len(results_test), len(x_df))
    x_scores = x_df['benefit_score'].iloc[:n] if 'benefit_score' in x_df.columns else pd.Series(np.nan, index=range(n))
    cf_scores = results_test['benefit_score'].iloc[:n]
    comparison_rows.append({'comparison': 'Causal forest vs GLMNet X-learner test output', 'comparison_basis': 'row_order_test_file_no_stable_member_id', 'n_compared': n, 'pearson_corr': safe_corr(cf_scores, x_scores, 'pearson'), 'spearman_corr': safe_corr(cf_scores, x_scores, 'spearman'), 'top_decile_overlap': top_overlap(cf_scores, x_scores, 0.10), 'top_20pct_overlap': top_overlap(cf_scores, x_scores, 0.20)})

if 'current_risk_score' in results_test.columns:
    comparison_rows.append({'comparison': 'Causal forest vs current risk score test output', 'comparison_basis': 'test_set_direct_columns', 'n_compared': len(results_test), 'pearson_corr': safe_corr(results_test['benefit_score'], results_test['current_risk_score'], 'pearson'), 'spearman_corr': safe_corr(results_test['benefit_score'], results_test['current_risk_score'], 'spearman'), 'top_decile_overlap': top_overlap(results_test['benefit_score'], results_test['current_risk_score'], 0.10), 'top_20pct_overlap': top_overlap(results_test['benefit_score'], results_test['current_risk_score'], 0.20)})

consistency_summary = pd.DataFrame(comparison_rows)
save_csv(consistency_summary, PATHS['consistency_summary'])
display(consistency_summary)

## Analytical Task 6: Variable Importance And Explainability

In [ ]:
importances = getattr(cf_model, 'feature_importances_', np.full(x_train.shape[1], np.nan))
importance_df = pd.DataFrame({'feature': x_train.columns, 'importance': np.asarray(importances, dtype=float)}).sort_values('importance', ascending=False).reset_index(drop=True)
importance_df.insert(0, 'rank', np.arange(1, len(importance_df) + 1))
save_csv(importance_df, PATHS['variable_importance'])
display(importance_df.head(20))

plot_df = importance_df.head(15).sort_values('importance', ascending=True)
plt.figure(figsize=(8, 5.5))
plt.barh(plot_df['feature'], plot_df['importance'])
plt.xlabel('Causal forest importance')
plt.ylabel('Feature')
plt.title('Top Causal Forest HTE Split Features')
save_current_figure(PATHS['variable_importance_chart'])

profile_features = ['current_risk_score', 'percolator_utilization_score', 'percolator_clinical_score', 'percolator_sdoh_score', 'ed_visits_last_6m', 'admits_last_6m', 'total_cost_last_6m', 'behavioral_health_risk_flag', 'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag', 'utilities_insecurity_flag', 'dual_eligible']
profile_features = [feature for feature in profile_features if feature in results_test.columns]
top_decile_mask = results_test['hte_decile'] == 1
top_decile_profile = pd.DataFrame([
    {'feature': feature, 'top_hte_decile_mean_or_rate': pd.to_numeric(results_test.loc[top_decile_mask, feature], errors='coerce').mean(), 'other_deciles_mean_or_rate': pd.to_numeric(results_test.loc[~top_decile_mask, feature], errors='coerce').mean()}
    for feature in profile_features
])
top_decile_profile['difference'] = top_decile_profile['top_hte_decile_mean_or_rate'] - top_decile_profile['other_deciles_mean_or_rate']
top_decile_profile = top_decile_profile.sort_values('difference', key=lambda s: s.abs(), ascending=False)
save_csv(top_decile_profile, PATHS['top_decile_profile'])
display(top_decile_profile)

## Analytical Task 7: Business Value Assessment

In [ ]:
targeting_summary = decile_summary[['hte_decile', 'n', 'avg_benefit_score', 'observed_ed_rate', 'treatment_pct']].copy()
if 'avg_current_risk_score' in decile_summary.columns:
    targeting_summary['avg_current_risk_score'] = decile_summary['avg_current_risk_score']
targeting_summary['cumulative_members'] = targeting_summary['n'].cumsum()
targeting_summary['cumulative_expected_ed_reductions'] = (targeting_summary['avg_benefit_score'] * targeting_summary['n']).cumsum()
save_csv(targeting_summary, PATHS['targeting_summary'])
display(targeting_summary)

## Analytical Task 8: Client Perspective

The causal forest model should initially be presented as a challenger and subgroup-discovery model. Its main value is estimating heterogeneous treatment effects directly and helping identify high-benefit member subgroups. Final operational use would require live-data validation, overlap review, treatment-effect uncertainty review, and monitoring.

## Recommendation

Use the generated outputs to decide whether causal forest should be presented as a primary prioritization model, a challenger model, or a subgroup-discovery tool. The default recommendation is to treat it as a challenger and subgroup-discovery framework until validated on live data.

## Presentation Summary

A presentation based on this notebook can be organized around the business problem, why causal forest is useful for HTE, data review, causal forest diagnostics, treatment-effect results, HTE deciles, framework consistency checks, explainability, business value, limitations, and recommendation.

## Reproducibility

Primary notebook: `Code/PRISM_Causal_Forest_Modeling_Workflow.ipynb`

Output folder: `Outputs/Causal-Forests/Python`

Seed: `123`

After this notebook is run successfully, the generated CSV tables and charts can be used to write `PRISM_Causal_Forest_Modeling_README.md`.